# Creating Images

So far we only sent text and files *into* a model and got text back. Models can also generate images for us.

In this notebook we let the model create a picture about AI engineering, and we look at every option OpenAI gives us for generating images.

Docs:
- [Image generation guide](https://developers.openai.com/api/docs/guides/image-generation)
- [Images API reference](https://developers.openai.com/api/docs/api-reference/images)

## The options at a glance

There are three ways to create images, from simplest to most flexible:

1. **Images API** (`client.images.generate`) - the direct, one-call way to turn a prompt into a picture.
2. **Responses API with the `image_generation` tool** - the model decides when to draw, so you can describe the image in a normal conversation and keep editing it across turns.
3. **Image editing** (`client.images.edit`) - start from an existing image (or several) and change it.

For most cases the Images API is enough. Use the Responses API when image creation is part of a larger chat, and use editing when you already have a picture to work from.

## Setup

In [ ]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

## Defining the model

Image generation uses a dedicated image model, not the text models from the earlier notebooks.

In [ ]:
IMAGE_MODEL = "gpt-image-1"

## A small helper to save and show images

The model returns the image as a base64 string. This helper decodes it, writes it to the `files/` folder, and shows it right in the notebook, so we don't repeat that code in every example.

In [ ]:
def save_and_show(image_base64, filename):
    image_path = Path("files") / filename
    image_path.write_bytes(base64.b64decode(image_base64))
    display(Image(filename=str(image_path)))
    return image_path

## Our prompt

A good image prompt describes the subject, the style, and the mood. The more concrete you are, the closer the result matches what you had in mind.

Let's be creative and describe an image about AI engineering.

In [ ]:
PROMPT = (
    "A friendly robot engineer sitting at a glowing desk, "
    "connecting colorful cables between a large language model and a small app, "
    "sticky notes that say 'prompt', 'eval', and 'ship' on the wall behind it. "
    "Clean modern flat illustration, soft blue and orange colors, friendly and optimistic mood."
)

## Option 1 - Images API

This is the most direct way: one call, prompt in, image out.

- `size` sets the resolution. `"auto"` lets the model choose, or pick `"1024x1024"`, `"1536x1024"` (landscape), or `"1024x1536"` (portrait).
- `quality` trades quality for speed and cost: `"low"`, `"medium"`, or `"high"`.

We use a smaller quality here to keep the call cheap and fast.

In [ ]:
response = client.images.generate(
    model=IMAGE_MODEL,
    prompt=PROMPT,
    size="1024x1024",
    quality="low",
)

save_and_show(response.data[0].b64_json, "ai-engineering.png")

## Option 2 - Responses API with the image_generation tool

Here we don't call an image endpoint directly. We give a normal text model a tool that lets it draw, and it decides to use it.

The big advantage: image creation becomes part of a conversation. We can keep talking and ask for changes, and the model remembers the previous image.

The generated image comes back inside the response output, in an `image_generation_call` item.

In [ ]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input=f"Create an image for a blog post about AI engineering. {PROMPT}",
    tools=[{"type": "image_generation", "quality": "low"}],
)

image_calls = [item for item in response.output if item.type == "image_generation_call"]

save_and_show(image_calls[0].result, "ai-engineering-from-chat.png")

### Editing it in the conversation

Because the image lives inside the conversation, we can ask for a change by passing `previous_response_id`. The model keeps the previous picture and only changes what we asked for.

In [ ]:
follow_up = client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,
    input="Add a small coffee cup on the desk and make the background a bit darker.",
    tools=[{"type": "image_generation", "quality": "low"}],
)

image_calls = [item for item in follow_up.output if item.type == "image_generation_call"]

save_and_show(image_calls[0].result, "ai-engineering-edited.png")

## Option 3 - Editing an existing image

Sometimes you already have an image and just want to change it. `client.images.edit` takes one or more input images plus a prompt.

Here we reuse the picture we generated in Option 1 and ask for a new variation of it.

In [ ]:
response = client.images.edit(
    model=IMAGE_MODEL,
    image=open("files/ai-engineering.png", "rb"),
    prompt="Turn this into a nighttime scene with a starry sky outside the window.",
    size="1024x1024",
    quality="low",
)

save_and_show(response.data[0].b64_json, "ai-engineering-night.png")

## Things worth remembering

- **Images cost more than text.** You pay per generated image, and higher `quality` and bigger `size` cost more. Start with low quality while you iterate on the prompt.
- **Prompt like you describe a photo.** Name the subject, the style (flat illustration, photo, watercolor), the colors, and the mood.
- **Pick the option that fits the job:** Images API for a quick one-off picture, Responses API when drawing is part of a chat you want to keep editing, and `images.edit` when you start from an existing image.
- **The result is base64.** You always decode it and save it yourself, like our small helper does.